# Per-video GCaMP analysis pipeline

This notebook analyzes every Suite2p video below `EXPERIMENT_ROOT` independently. It writes each video's metrics, figures, and a versioned `*_analysis_summary.json` artifact for later comparison. It does not configure groups, aggregate folders, register longitudinal recordings, or compare treatments.

In [ ]:
from pathlib import Path
import sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'gcamp_analysis').is_dir() else WORKING_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.artifacts import discover_summary_artifacts

In [ ]:
# Analysis settings
SMOOTH_SIGMA = 1.0

config_path = PROJECT_ROOT / 'config' / 'notebook_config.yaml'
config = load_config(config_path)
config.setdefault('traces', {})['smooth_sigma'] = SMOOTH_SIGMA
grouping_config = config.setdefault('grouping', {})
grouping_config['strategies'] = ['combined', 'light-evoked']
print(f'Config: {config_path}')
print(f'Trace smoothing sigma: {SMOOTH_SIGMA}')

### Grouping: Correlated Firing with STTC and Pearson correlation

This strategy multiplies a lagged Pearson-correlation similarity matrix by an STTC spike-timing similarity matrix, then clusters the combined result.

- `RUN_COMBINED_GROUPING`: Enables or disables this strategy. Example: `True` runs correlated-firing grouping; `False` skips it.
- `PEARSON_MAX_LAG_FRAMES`: Largest positive or negative trace offset considered when finding the maximum Pearson correlation. Example: `5` searches lags from -5 through +5 frames. At 3 Hz, five frames is about 1.67 seconds.
- `STTC_DT_SECONDS`: Coincidence window, in seconds, used by the spike-time tiling coefficient. Example: `1.75` treats spikes within 1.75 seconds as temporally associated. Smaller values require tighter timing.
- `CLUSTER_LINKAGE_METHOD`: Hierarchical-clustering linkage rule. Example: `'average'` uses the average distance between members of two candidate clusters; `'complete'` uses their greatest distance and generally favors tighter groups.
- `CLUSTER_CRITERION`: SciPy rule used to cut the hierarchy. Example: `'distance'` applies `CLUSTER_THRESHOLD` as a maximum cophenetic distance.
- `CLUSTER_THRESHOLD`: Cut value supplied to the selected clustering criterion. With `'distance'`, lower values produce stricter, usually smaller groups. Example: `0.5`.
- `MIN_GROUP_SIZE`: Minimum neurons required to retain a cluster. Example: `2` discards single-neuron clusters.

**Example:** For 3 Hz recordings, `PEARSON_MAX_LAG_FRAMES = 5`, `STTC_DT_SECONDS = 1.75`, and `CLUSTER_THRESHOLD = 0.5` allow modest temporal offsets while retaining only combined-similarity clusters with at least two neurons.

In [ ]:
RUN_COMBINED_GROUPING = False
PEARSON_MAX_LAG_FRAMES = 5
STTC_DT_SECONDS = 1.75
CLUSTER_LINKAGE_METHOD = 'average'
CLUSTER_CRITERION = 'distance'
CLUSTER_THRESHOLD = 0.5
MIN_GROUP_SIZE = 2

combined_config = grouping_config.setdefault('combined', {})
combined_config['enabled'] = RUN_COMBINED_GROUPING
combined_config.setdefault('corr', {})['max_lag'] = PEARSON_MAX_LAG_FRAMES
combined_config.setdefault('sttc', {})['dt'] = STTC_DT_SECONDS
combined_config.setdefault('cluster', {}).update({
    'linkage_method': CLUSTER_LINKAGE_METHOD,
    'cluster_criterion': CLUSTER_CRITERION,
    'cluster_param': CLUSTER_THRESHOLD,
    'min_group_size': MIN_GROUP_SIZE,
})

### Grouping: Light-evoked Response

This strategy groups neurons by how many scheduled light pulses evoke an ON or OFF response. Frame-based values depend on the recording frame rate.

- `RUN_LIGHT_EVOKED_GROUPING`: Enables or disables this strategy. Example: `True` runs light-evoked grouping.
- `LIGHT_START_FRAME`: Frame of the first regularly spaced light pulse. Example: `30` is 10 seconds into a 3 Hz recording. Set this and `LIGHT_INTERVAL_FRAMES` to `None` when using `LIGHT_SCHEDULE`.
- `LIGHT_INTERVAL_FRAMES`: Frames between regularly spaced pulses. Example: `30` means one pulse every 10 seconds at 3 Hz.
- `LIGHT_BIN_SIZE_FRAMES`: Width of the pulse-aligned window in which fluorescence-derivative peaks are detected. Example: `2` examines two frames beginning at each pulse.
- `LIGHT_RESPONSE_WINDOW_FRAMES`: Maximum number of frames after a pulse in which a detected spike can be matched to that pulse. Example: `10` allows responses through 10 frames after onset.
- `LIGHT_PROMINENCE`: Optional minimum derivative-peak prominence. Example: `None` accepts peaks without prominence filtering; `0.2` rejects peaks below 0.2 prominence units.
- `LIGHT_SCHEDULE`: Optional explicit list of pulse-onset frames for irregular stimulation. Example: `[30, 65, 93, 116]`. It is used only when both start and interval are `None`.

**Regular example:** `LIGHT_START_FRAME = 30`, `LIGHT_INTERVAL_FRAMES = 30`, and `LIGHT_BIN_SIZE_FRAMES = 2` describe regularly spaced pulses starting at frame 30.

**Irregular example:** Set `LIGHT_START_FRAME = None`, `LIGHT_INTERVAL_FRAMES = None`, and `LIGHT_SCHEDULE = [30, 65, 93, 116]`.

In [ ]:
RUN_LIGHT_EVOKED_GROUPING = True
LIGHT_START_FRAME = 30
LIGHT_INTERVAL_FRAMES = 30
LIGHT_BIN_SIZE_FRAMES = 2
LIGHT_RESPONSE_WINDOW_FRAMES = 10
LIGHT_PROMINENCE = None
LIGHT_SCHEDULE = None  # Set start and interval to None to use an explicit pulse-frame list

light_evoked_config = grouping_config.setdefault('light-evoked', {})
light_evoked_config.update({
    'enabled': RUN_LIGHT_EVOKED_GROUPING,
    'start': LIGHT_START_FRAME,
    'interval': LIGHT_INTERVAL_FRAMES,
    'bin_size': LIGHT_BIN_SIZE_FRAMES,
    'response_window': LIGHT_RESPONSE_WINDOW_FRAMES,
    'prominence': LIGHT_PROMINENCE,
    'schedule': LIGHT_SCHEDULE,
})

enabled_grouping = [
    name for name in grouping_config['strategies']
    if grouping_config[name]['enabled']
]
print(f'Grouping strategies: {enabled_grouping or "none"}')

## Model Selection

### Hugging Face

The ROI and spike classifiers are loaded from a pinned Hugging Face repository revision. Downloads are cached locally after the first successful load.

- `HF_REPO_ID`: Hugging Face repository containing the classifier folders. Example: `'mmzinn12/gcamp-analysis-models'`.
- `HF_REVISION`: Immutable repository version to load, given as a full commit hash or release tag. Pinning the revision makes analyses reproducible. Example: `'c566b58e7dd1f63934f65566ac525cf12db914f5'`. Do not use a mutable branch such as `'main'` or `'master'`.
- `ROI_MODEL_NAME`: Folder name for the ROI-quality classifier inside the repository's `roi/` directory. Example: `'3hz_invivo_base'` selects `roi/3hz_invivo_base`. Choose a model compatible with the recording frame rate and preparation.
- `SPIKE_MODEL_NAME`: Folder name for the spike classifier inside the repository's `spike/` directory. Example: `'3hz_invivo_base'` selects `spike/3hz_invivo_base`.
- `ROI_MODEL_FOLDER`: Repository-relative ROI folder assembled from `ROI_MODEL_NAME`. It is derived automatically and normally should not be edited directly.
- `SPIKE_MODEL_FOLDER`: Repository-relative spike folder assembled from `SPIKE_MODEL_NAME`. It is derived automatically and normally should not be edited directly.

**Example:** The values below load both 3 Hz in-vivo classifiers from the same pinned repository commit, ensuring the selected models do not change between analysis runs.

In [ ]:
# Hugging Face model selection
HF_REPO_ID = 'mmzinn12/gcamp-analysis-models'
HF_REVISION = 'c566b58e7dd1f63934f65566ac525cf12db914f5'
ROI_MODEL_NAME = '3hz_invivo_base'
SPIKE_MODEL_NAME = '3hz_invivo_base'
ROI_MODEL_FOLDER = f'roi/{ROI_MODEL_NAME}'
SPIKE_MODEL_FOLDER = f'spike/{SPIKE_MODEL_NAME}'

model_source = {
    'source': 'huggingface',
    'repo_id': HF_REPO_ID,
    'revision': HF_REVISION,
}
roi_model, roi_cfg = load_model(
    model_source, which='roi', model_folder=ROI_MODEL_FOLDER
)
spike_model, spike_cfg = load_model(
    model_source, which='spike', model_folder=SPIKE_MODEL_FOLDER
)
models = {
    'roi': roi_model,
    'roi_config': roi_cfg,
    'spike': spike_model,
    'spike_config': spike_cfg,
}
runner = VideoPipelineRunner.build(config, models)
print(f'ROI model:   {type(roi_model).__name__}')
print(f'Spike model: {type(spike_model).__name__}')

## Run 

In [ ]:
EXPERIMENT_ROOT = Path(r'C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710')  # change for each analysis batch
DRY_RUN = False  # False writes per-video outputs
assert EXPERIMENT_ROOT.exists(), f'Experiment root not found: {EXPERIMENT_ROOT}'

tree = ExperimentTreeBuilder(is_video_dir=is_video_dir).build(EXPERIMENT_ROOT)
print_tree(tree)

In [ ]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
    dry_run=DRY_RUN,
    analysis_metadata={
        'config': config,
        'sensor_type': config.get('traces', {}).get('sensor_type'),
    },
)
processor.process_videos(tree, verbose=True)

In [ ]:
processed = [node for node in tree.iter_nodes() if node.payload is not None]
print(f'Analyzed {len(processed)} video(s).')
if DRY_RUN:
    print('Dry run complete: no files were written.')
else:
    summaries = discover_summary_artifacts(EXPERIMENT_ROOT)
    print(f'Wrote {len(summaries)} comparison-ready video summary artifact(s).')